In [ ]:
import os
from openai import OpenAI
import pandas as pd

client = OpenAI(
    # This is the default and can be omitted
    api_key=""
)

In [3]:
df = pd.read_csv("./ADRD_Article_Table.csv", encoding="latin1", index_col=0)
df_titles_abstract = df[['Title', 'Abstract']]
df['Inclusion Decision'] = "NA"

In [4]:


SR_title = "Digital Interventions Targeting Modifiable Risk Factors for Dementia: A Scoping Review"

incl_criteria = (
    "Studies were included if they were published from 2019 to the present and written in English. "
    "Eligible studies evaluated an intervention with a digital health component targeting modifiable behavioral risk or protective factors "
    "for Alzheimer's disease and related dementias (ADRD). The target population included individuals not yet diagnosed with ADRD. "
    "Interventions targeting individuals with mild cognitive impairment (a stage in the normal aging process and a known risk factor with preventive potential) were included. "
    "Studies were eligible if they assessed feasibility (e.g., recruitment or data collection approaches), acceptability (e.g., participant evaluation of intervention content), "
    "psychosocial outcomes (e.g., changes in attitudes or self-efficacy toward a behavior), behavioral outcomes (e.g., changes in actual behavioral performance), or health outcomes."
)

excl_criteria = (
    "Studies were excluded if they were published before 2019 or written in a language other than English. "
    "Studies were also excluded if participants included individuals with ADRD or a related diagnosis, "
    "or if participants had amnestic cognitive impairment (representing atypically severe decline for age). "
    "Interventions evaluating diagnostic or pharmaceutical to ADRD prevention approaches rather than behavioral interventions were excluded."
    "Studies that did not evaluate interventions were excluded, as were cross-sectional studies, cohort studies, meta-analyses, reviews, systematic reviews."
    "Studies that do not collect or analyze empirical data (e.g., protocols, studies using simulated data) were excluded."
)


def build_prompt(title: str, abstract: str) -> str:

    return (
        f"You are screening articles for inclusion in the scoping review titled '{SR_title}'. "
        "Using the inclusion and exclusion criteria provided below, decide whether the screened article "
        "should be included based on its title and abstract. "
        "If the article appears relevant to the review’s theme and does not clearly meet any exclusion criteria, lean toward inclusion. "
        "Respond only with either 'Include' or 'Exclude'.\n\n"
        f"Systematic Review Title: {SR_title}\n\n"
        f"Inclusion Criteria:\n{incl_criteria}\n\n"
        f"Exclusion Criteria:\n{excl_criteria}\n\n"
        f"Article Title: {title}\n"
        f"Article Abstract: {abstract}"
    )


response_template = (
    "Systematic Review Title: {SR_title}\n\n"
    "Inclusion Criteria:\n{incl_criteria}\n\n"
    "Exclusion Criteria:\n{excl_criteria}\n\n"
    "Article Title: {title}\n"
    "Article Abstract: {abstract}\n\n"
    "Decision: {decision}"
)



In [84]:
print(build_prompt(df["Title"][0], df["Abstract"][0]))


You are screening articles for inclusion in the scoping review titled 'Digital Interventions Targeting Modifiable Risk Factors for Dementia: A Scoping Review'. Using the inclusion and exclusion criteria provided below, decide whether the screened article should be included based on its title and abstract. If the article appears relevant to the review’s theme and does not clearly meet any exclusion criteria, lean toward inclusion. Respond only with either 'Include' or 'Exclude'.

Systematic Review Title: Digital Interventions Targeting Modifiable Risk Factors for Dementia: A Scoping Review

Inclusion Criteria:
Studies were included if they were published from 2019 to the present and written in English. Eligible studies evaluated an intervention with a digital health component targeting modifiable behavioral risk or protective factors for Alzheimer's disease and related dementias (ADRD). The target population included individuals not yet diagnosed with ADRD. Interventions targeting indiv

In [30]:
def assess_inclusion_api(title, abstract):

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful assistant to academic public health researchers."},
            {"role": "user", "content": build_prompt(title, abstract)}
        ],
        max_tokens=50,
        temperature=0.0,
    )
    return response

,

''

In [34]:
df = pd.read_csv("./ADRD_Article_Table.csv", encoding="latin1", index_col=0)
df_titles_abstract = df[['Title', 'Abstract']]
df['Inclusion Decision'] = "NA"

In [58]:
def results_title_abstract_screening(df):
    for index, row in df.iterrows():
        title = row['Title']
        abstract = row['Abstract']
        decision = assess_inclusion_api(title, abstract).choices[0].message.content
        df.at[index, 'Inclusion Decision'] = decision
        print(f"Article {index + 1}/{len(df)}: {decision}")
    return df

In [ ]:
# df_results = results_title_abstract_screening(df_titles_abstract)

C:\Users\Rodrigo\AppData\Local\Temp\ipykernel_17656\2095543515.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.at[index, 'Inclusion Decision'] = decision


Article 1/1079: Exclude
Article 2/1079: Exclude
Article 3/1079: Exclude
Article 4/1079: Exclude
Article 5/1079: Exclude
Article 6/1079: Include
Article 7/1079: Exclude
Article 8/1079: Exclude
Article 9/1079: Exclude
Article 10/1079: Exclude
Article 11/1079: Include
Article 12/1079: Exclude
Article 13/1079: Exclude
Article 14/1079: Exclude
Article 15/1079: Exclude
Article 16/1079: Exclude
Article 17/1079: Exclude
Article 18/1079: Exclude
Article 19/1079: Exclude
Article 20/1079: Exclude
Article 21/1079: Exclude
Article 22/1079: Exclude
Article 23/1079: Exclude
Article 24/1079: Exclude
Article 25/1079: Include
Article 26/1079: Exclude
Article 27/1079: Exclude
Article 28/1079: Exclude
Article 29/1079: Exclude
Article 30/1079: Exclude
Article 31/1079: Exclude
Article 32/1079: Exclude
Article 33/1079: Exclude
Article 34/1079: Exclude
Article 35/1079: Exclude
Article 36/1079: Exclude
Article 37/1079: Exclude
Article 38/1079: Exclude
Article 39/1079: Exclude
Article 40/1079: Exclude
Article 4

In [9]:
# df_results.to_csv("./ADRD_Article_Table_Screened_LLM.csv", encoding="latin1")
df_results = pd.read_csv("./ADRD_Article_Table_Screened_LLM.csv", encoding="latin1", index_col=0)

In [10]:
df_results[df_results['Inclusion Decision'] == "Include"]

,Title,Abstract,Inclusion Decision
Index,,,
5,Cognitive Training with Neurofeedback Using fN...,This study examined the effects of a 4-week co...,Include
10,Attitudes of At-Risk Older Adults about Preven...,OBJECTIVES: Prevention of cardiovascular disea...,Include
24,Efficacy of an mHealth Behavior Change Interve...,Background: Insufficient physical activity (PA...,Include
52,Effects of Virtual Reality Physical and Cognit...,BACKGROUND: Virtual reality (VR) technology ha...,Include
72,Results of the Italian RESILIEN-T Pilot Study:...,(1) Background: The RESILIEN-T system addresse...,Include
...,...,...,...
1059,Intervention of Computer-Assisted Cognitive Tr...,Objective Investigate the impact of combined c...,Include
1060,Short-Term Computerized Cognitive Training Doe...,"BACKGROUND: Older adults, especially those abo...",Include
1063,Effects of Computerized Cognitive Training on ...,BACKGROUND: People with mild cognitive impairm...,Include


In [ ]:
print(df_results[['Title', 'Abstract']].head(10))

                                                   Title  \
Index                                                      
0      Examining the Feasibility and Acceptability of...   
1      The Use of Digital Touch Screen Technology to ...   
2      How to Advance the Pharmacological Management ...   
3      Evaluating the Utility of Digital Phenotyping ...   
4      Use of Digital Health Technologies for Dementi...   
5      Cognitive Training with Neurofeedback Using fN...   
6      Effect of Reductions in Amyloid Levels on Cogn...   
7      Harnessing Passive Pulsatile Shear Stress for ...   
8      Sodium-Calcium Exchanger Isoform-3 Targeted Wi...   
9      The Alzheimer's Disease Neuroimaging Initiativ...   

                                                Abstract  
Index                                                     
0      Introduction The use of technology to deliver ...  
1      Background Using technology to deliver psychos...  
2      Cognitive impairment is a common non

: 

In [11]:
df_ai_results = df_results[df_results['Inclusion Decision'] == "Include"]

## Compare the AI selection to our selection

In [7]:
df_human_results = pd.read_csv("C:/Users/Rodrigo/Desktop/ADRD Scoping Review/Final/included.csv", encoding="latin1", index_col=0)
df_human_results

,Title,Abstract,Keywords,Author,Year,Journal,DOI,Source,"Relevant (0, 1)"
Index,,,,,,,,,
5,Cognitive Training with Neurofeedback Using fN...,This study examined the effects of a 4-week co...,"*Cognitive Dysfunction/therapy,*Neurofeedback/...","Acevedo, Bianca P. and Dattatri, Novia and Le,...",2022,International journal of environmental researc...,10.3390/ijerph19095531,PubMed,NaN
20,"mHealth Apps for Dementia, Alzheimer Disease, ...",BACKGROUND: Lifestyle behaviors including exer...,"*Alzheimer Disease/psychology/therapy,*Dementi...","Ali, Suad and Alizai, Hira and Hagos, Delal Je...",2024,JMIR mHealth and uHealth,10.2196/50186,PubMed,NaN
31,A Scoping Survey to Inform Design of Digital D...,Background: Digital dementia risk reduction in...,"Alzheimer's disease,cognition,dementia,prevent...","Amos, Jessica G. and Zheng, Lidan and Eramudug...",2024,JOURNAL OF ALZHEIMERS DISEASE REPORTS,10.3233/ADR-230160,Web of Science,NaN
32,MyCOACH (COnnected Advice for Cognitive Health...,INTRODUCTION: Digital health interventions are...,*Cognitive Dysfunction/prevention & control/ps...,"Amos, Jessica G. and Zheng, Lidan and Eramudug...",2023,BMJ open,10.1136/bmjopen-2023-075015,PubMed,NaN
52,Effects of Virtual Reality Physical and Cognit...,BACKGROUND: Virtual reality (VR) technology ha...,"Alzheimer's disease,mild cognitive impairment,...","Baldimtsi, Eleni and Mouzakidis, Christos and ...",2023,Journal of Alzheimer's disease reports,10.3233/ADR-230099,PubMed,NaN
...,...,...,...,...,...,...,...,...,...
1032,Predicting Adherence to Computer-Based Cogniti...,BACKGROUND: Cognitive impairment and dementia ...,"*Cognitive Dysfunction/psychology/therapy,*Dee...","Singh, Ankita and Chakraborty, Shayok and He, ...",2024,JMIR aging,10.2196/53793,pubmed,NaN
1033,Comparing Five Activities to Improve Quality o...,BACKGROUND: People with mild cognitive impairm...,"dementia,feasibility studies,geriatrics,intern...","Smith, Glen and Chandler, Melanie and Ball, Co...",2020,Comparing Five Activities to Improve Quality o...,10.25302/12.2020.CER.1306-01897IC,pubmed,NaN
1038,"The Effect of Computerized Cognitive Training,...",Background: Slower walking is associated with ...,NaN,"Stein, Ryan G. and Ten Brinke, Lisanne F. and ...",2024,JOURNAL OF ALZHEIMERS DISEASE REPORTS,10.3233/ADR-230206,webofscience,NaN


In [12]:
indices_to_review = [rec for rec in df_ai_results.index if rec not in df_human_results.index]
print(len(indices_to_review ))

80


In [13]:
articles_to_review = df.loc[indices_to_review]
articles_to_review.to_csv("./ADRD_Articles_to_Review.csv", encoding="latin1")

## Select Random 20 records:

In [21]:
sample_df = df_human_results.sample(n=20, random_state=42).sort_index()
sample_df['Citation'] = sample_df.apply(lambda row: f"{row['Author'].split(',')[0]} ({row['Year']})", axis=1)
sample_df
sample_df.to_csv("./data_extraction_validation_set.csv", encoding="latin1")

In [22]:
sample_df = df_human_results.sample(n=20, random_state=49).sort_index()
sample_df['Citation'] = sample_df.apply(lambda row: f"{row['Author'].split(',')[0]} ({row['Year']})", axis=1)
sample_df
sample_df.to_csv("./data_extraction_validation_set_2.csv", encoding="latin1")
